Christian Thede
CNN Model

Notes: 
1. Used AI for mounting and dealing with colab and google drive (and still had a ton of issues)
2. 

In [ ]:
# import ml libraries
import tensorflow as tf
import skimage
from tensorflow.keras import layers

import numpy as np
from tensorflow import keras

from sklearn.model_selection import train_test_split

from skimage import io
from skimage.transform import resize

from torchvision import datasets, transforms

from pathlib import Path

NUMS_DIR = Path("/content/drive/MyDrive/Colab Notebooks/Nums")
OUTPUT_DIR = NUMS_DIR.parent / "Output"

print("NUMS_DIR:", NUMS_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

Mounted at /content/drive


In [ ]:
# Google Drive mount for Colab
import os
import shutil

mountpoint = "/content/drive"

from google.colab import drive
drive.mount(mountpoint, force_remount=True)

### Data

In [6]:
IMAGE_SIZE = (28, 28)
NUM_CLASSES = 10

train_ds = datasets.MNIST("./MNIST", train=True, transform=transforms.ToTensor(), download=True)
X_train = np.array([img.numpy().transpose(1, 2, 0) for img, _ in train_ds], dtype=np.float32)
y_train = np.array([lbl for _, lbl in train_ds], dtype=np.int64)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, train_size=0.9, random_state=42, stratify=y_train
)

nums_dir = Path("/content/drive/MyDrive/Colab Notebooks/Nums")
files = sorted(nums_dir.glob("*.png"))
X_test = np.array(
    [resize(io.imread(str(p), as_gray=True), IMAGE_SIZE, anti_aliasing=True)[..., None] for p in files],
    dtype=np.float32,
)
y_test = np.array([int(p.stem[0]) for p in files], dtype=np.int64)

y_train_oh = keras.utils.to_categorical(y_train, NUM_CLASSES)
y_val_oh = keras.utils.to_categorical(y_val, NUM_CLASSES)
y_test_oh = keras.utils.to_categorical(y_test, NUM_CLASSES)

print(X_train.shape, X_val.shape, X_test.shape)

(54000, 28, 28, 1) (6000, 28, 28, 1) (30, 28, 28, 1)


### Model Building

In [9]:
# build model
model = keras.Sequential(
    [
        layers.Input(shape=(IMAGE_SIZE[1], IMAGE_SIZE[0], INPUT_CHANNELS)),
        layers.Conv2D(filters=5, kernel_size=3, padding="same", activation="relu"),
        layers.MaxPooling2D(pool_size=2),
        layers.Conv2D(filters=8, kernel_size=3, padding="same", activation="tanh"),
        layers.MaxPooling2D(pool_size=2),
        layers.Flatten(),
        layers.Dense(300, activation="tanh"),
        layers.Dense(NUM_CLASSES, activation="softmax"),
    ]
)

model.summary()

NameError: name 'INPUT_CHANNELS' is not defined

### Model Compilation

In [ ]:
model.compile(
optimizer=keras.optimizers.Adam(learning_rate=3e-4),
loss="categorical_crossentropy",
metrics=["accuracy"],
)

callbacks = [
keras.callbacks.EarlyStopping(
monitor="val_loss", patience=20, restore_best_weights=True
)
]

### Model Training

In [ ]:
# train
history = model.fit(
    X_train,
    y_train_oh,
    epochs=1,
    batch_size=8,
    validation_data=(X_val, y_val_oh),
    callbacks=callbacks,
    verbose=1,
)

### Model Evaluation

In [ ]:
# evaluate
train_loss, train_acc = model.evaluate(X_train, y_train_oh, verbose=0)
val_loss, val_acc = model.evaluate(X_val, y_val_oh, verbose=0)
test_loss, test_acc = model.evaluate(X_test, y_test_oh, verbose=0)

print(f"Train loss: {train_loss:.4f} | Train accuracy: {train_acc:.4f}")
print(f"Val loss:   {val_loss:.4f} | Val accuracy:   {val_acc:.4f}")
print(f"Test loss:  {test_loss:.4f} | Test accuracy:  {test_acc:.4f}")
print(f"Set sizes -> train: {len(X_train)}, val: {len(X_val)}, test: {len(X_test)}")

# save model
model_path = OUTPUT_DIR / "digit_cnn.keras"
model.save(model_path)
print("Saved model to", model_path)

### Prediction

In [ ]:
# predictions
probs = model.predict(X_test, verbose=0)
y_pred = np.argmax(probs, axis=1)

# sample views
for i in range(min(12, X_test.shape[0])):
    print(f"true={y_test[i]} pred={y_pred[i]} probs={np.round(probs[i], 3)}")

### Visualizations

In [ ]:
# visualizations
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="train")
plt.plot(history.history["val_accuracy"], label="val")
plt.axhline(test_acc, linestyle="--", label=f"test ({test_acc:.2f})")
plt.title("Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="train")
plt.plot(history.history["val_loss"], label="val")
plt.axhline(test_loss, linestyle="--", label=f"test ({test_loss:.2f})")
plt.title("Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.tight_layout()
plt.show()
